# Harbourline Finance — Credit Risk Analysis

## Data Understanding

### Business Problem

Harbourline Finance currently approves applicants with a bureau score of 540 or above.

The objective of this analysis is to determine whether a statistical Probability of Default (PD) model can improve credit decision-making by:

- identifying characteristics associated with default;
- predicting borrower probability of default;
- estimating expected credit losses;
- identifying a profit-maximising approval cut-off;
- assessing the risk of the Q1 2025 applicant population; and
- translating the findings into a management-ready Power BI dashboard.

### Analytical Workflow

1. Data understanding
2. Data cleaning and exploratory analysis
3. SQL analysis
4. PD modelling and validation
5. Expected loss and profitability analysis
6. Power BI dashboard
7. Business recommendations

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
db_path =  (r"C:\Users\Administrator\Desktop\harborline_credit_risk\data\raw\harbourline_credit.db")
conn = sqlite3.connect(db_path)

print("Database Connection Successful...")

Database Connection Successful...


In [3]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table' 
    ORDER by name;
    """,
    conn
)

tables

,name
0,applications_2025q1
1,assumptions
2,data_dictionary
3,loans_2023_2024


#### checking the historical table

In [4]:

loans = pd.read_sql_query(
    "SELECT * FROM loans_2023_2024",
    conn
)

In [5]:
loans.head()

,loan_id,application_month,age,employment_type,employment_years,annual_income,home_ownership,bureau_score,credit_utilisation,delinquencies_2y,...,loan_purpose,loan_amount,term_months,debt_to_income,channel,region,interest_rate,default_12m,ead_at_default,recovery_amount
0,HL100001,2023-01-01,34,Salaried,6.0,32200.0,Own,690,0.639,0,...,Vehicle,13500.0,24,0.504,Online,Rural,0.1529,0,0.0,0.0
1,HL100002,2023-01-01,43,Self-employed,0.8,20700.0,Rent,632,0.758,0,...,Debt consolidation,7500.0,24,0.278,Branch,Inland,0.1787,0,0.0,0.0
2,HL100003,2023-01-01,34,Informal,1.4,17100.0,Rent,630,0.391,0,...,Home improvement,2500.0,24,0.371,Online,Coastal,0.1770,0,0.0,0.0
3,HL100004,2023-01-01,29,Salaried,3.1,33700.0,Mortgage,622,0.290,0,...,Education,6100.0,48,0.137,Branch,Metro North,0.1803,0,0.0,0.0
4,HL100005,2023-01-01,19,Self-employed,2.5,48100.0,Rent,574,0.476,3,...,Business,20700.0,48,0.597,Branch,Coastal,0.2652,0,0.0,0.0


In [6]:
loans.info()

<class 'pandas.DataFrame'>
RangeIndex: 11582 entries, 0 to 11581
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   loan_id             11582 non-null  str    
 1   application_month   11582 non-null  str    
 2   age                 11582 non-null  int64  
 3   employment_type     11582 non-null  str    
 4   employment_years    11210 non-null  float64
 5   annual_income       11349 non-null  float64
 6   home_ownership      11582 non-null  str    
 7   bureau_score        11582 non-null  int64  
 8   credit_utilisation  11582 non-null  float64
 9   delinquencies_2y    11582 non-null  int64  
 10  inquiries_6m        11582 non-null  int64  
 11  loan_purpose        11582 non-null  str    
 12  loan_amount         11582 non-null  float64
 13  term_months         11582 non-null  int64  
 14  debt_to_income      11582 non-null  float64
 15  channel             11582 non-null  str    
 16  region         

In [7]:
loans.describe().T

,count,mean,std,min,25%,50%,75%,max
age,11582.0,38.468486,44.118614,0.0000,29.0000,36.0000,44.0000,999.0000
employment_years,11210.0,4.943684,3.487907,0.0000,2.4000,4.2000,6.7000,30.7000
annual_income,11349.0,34319.393779,18475.928860,3900.0000,21500.0000,30000.0000,42700.0000,208300.0000
bureau_score,11582.0,644.185892,51.017322,540.0000,606.0000,642.0000,679.0000,820.0000
credit_utilisation,11582.0,0.468845,0.209882,0.0000,0.3090,0.4530,0.6170,1.1310
delinquencies_2y,11582.0,0.737178,1.081194,0.0000,0.0000,0.0000,1.0000,9.0000
inquiries_6m,11582.0,1.196512,1.098182,0.0000,0.0000,1.0000,2.0000,7.0000
loan_amount,11582.0,9085.857365,6273.229233,1000.0000,4600.0000,7500.0000,11700.0000,40000.0000
term_months,11582.0,28.632361,11.703120,12.0000,24.0000,24.0000,36.0000,48.0000
debt_to_income,11582.0,0.341958,0.147870,0.0330,0.2290,0.3240,0.4380,0.8780


#### the current 2025 applications q1

In [8]:
applications = pd.read_sql_query(
    "SELECT * FROM applications_2025q1",
    conn
)

In [10]:
applications.shape

(3000, 18)

In [11]:
applications.head()

,application_id,application_month,age,employment_type,employment_years,annual_income,home_ownership,bureau_score,credit_utilisation,delinquencies_2y,inquiries_6m,loan_purpose,loan_amount,term_months,debt_to_income,channel,region,interest_rate
0,AP500001,2025-01-01,54,Salaried,3.1,39200.0,Mortgage,629,0.447,1,2,Debt consolidation,11700.0,36,0.206,Online,Metro South,0.1691
1,AP500002,2025-02-01,41,Salaried,10.1,15500.0,Rent,630,0.745,0,1,Debt consolidation,3700.0,24,0.401,Online,Metro North,0.1854
2,AP500003,2025-01-01,59,Salaried,5.5,26600.0,Own,604,0.492,2,1,Medical,8500.0,48,0.583,Branch,Coastal,0.2272
3,AP500004,2025-01-01,38,Salaried,12.7,49500.0,Own,725,0.021,1,2,Vehicle,21200.0,12,0.493,Online,Coastal,0.1238
4,AP500005,2025-02-01,42,Contract,12.8,27300.0,Mortgage,573,0.364,2,1,Medical,4800.0,36,0.167,Branch,Metro South,0.2795


In [12]:
applications.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   application_id      3000 non-null   str    
 1   application_month   3000 non-null   str    
 2   age                 3000 non-null   int64  
 3   employment_type     3000 non-null   str    
 4   employment_years    2884 non-null   float64
 5   annual_income       2943 non-null   float64
 6   home_ownership      3000 non-null   str    
 7   bureau_score        3000 non-null   int64  
 8   credit_utilisation  3000 non-null   float64
 9   delinquencies_2y    3000 non-null   int64  
 10  inquiries_6m        3000 non-null   int64  
 11  loan_purpose        3000 non-null   str    
 12  loan_amount         3000 non-null   float64
 13  term_months         3000 non-null   int64  
 14  debt_to_income      3000 non-null   float64
 15  channel             3000 non-null   str    
 16  region           